# SPINEPS on the 12 healthy Spine-Generic necks - clean vertebral-BODY (corpus) labels

Goal: get a learned vertebral-BODY label (corpus) **separate from the posterior arch**, to fix the C6/C7
body-isolation that canal-cut on TotalSpineSeg mishandles (the cause of the residual Cobb/slip endpoint
error). SPINEPS (Moller 2025, Apache-2.0) runs natively on sagittal T2w MRI and outputs:
- `seg-spine` (semantic, 14 structures): **corpus border = 49, endplate = 62**
- `seg-vert` (instance): vertebrae 1-25, IVDs 100+X, endplates 200+X

Downstream (local): per-vertebra body = `(seg-spine == 49) & (seg-vert == that vertebra)` -> run our
endplate-line Cobb/heights on it and compare to the canal-cut TSS results, especially at C6/C7.

Resumable; needs a GPU (Runtime -> T4 GPU). Input = the same `sciseg_healthy_pilot.zip` (12 healthy T2w).

In [ ]:
!nvidia-smi -L   # must list a GPU; if blank, Runtime -> change runtime type -> T4 GPU, then re-run

In [ ]:
# SPINEPS (PyPI). If this errors on deps, fall back to: 
#   !git clone --depth 1 https://github.com/Hendrik-code/spineps.git && pip install -e spineps
!pip install -q spineps 2>&1 | tail -4
import importlib, spineps; importlib.reload(spineps)
print('spineps', getattr(spineps, '__version__', '?'))
# Model weights auto-download on first `spineps sample` run.

In [ ]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
# Gotcha: if it says 'Mountpoint must not already contain files', run in a fresh cell:
#   !fusermount -u /content/drive 2>/dev/null; rm -rf /content/drive   then re-run this cell.

In [ ]:
import os, glob
!unzip -q -o /content/drive/MyDrive/sciseg_healthy_pilot.zip -d /content/cases
OUT = '/content/drive/MyDrive/spinegeneric_spineps'; os.makedirs(OUT, exist_ok=True)
cases = sorted(c for c in glob.glob('/content/cases/**/*.nii.gz', recursive=True) if 'seg' not in c.lower())
print(len(cases), 'input T2w cases')

In [ ]:
import os, glob, subprocess, shutil, time
OUT = '/content/drive/MyDrive/spinegeneric_spineps'

def seg_one(f):
    cmd = ['spineps', 'sample', '-ignore_bids_filter', '-ignore_inference_compatibility',
           '-i', f, '-model_semantic', 't2w', '-model_instance', 'instance']
    return subprocess.run(cmd, capture_output=True, text=True)

for i, f in enumerate(cases, 1):
    base = os.path.basename(f)[:-7]                       # strip .nii.gz
    if glob.glob(f'{OUT}/{base}*seg-vert*'):              # resumable
        print(f'[{i}/{len(cases)}] skip (done): {base}'); continue
    t = time.time(); print(f'[{i}/{len(cases)}] {base}', flush=True)
    r = seg_one(f)
    outs = (glob.glob('/content/cases/**/*seg-spine*.nii.gz', recursive=True) +
            glob.glob('/content/cases/**/*seg-vert*.nii.gz', recursive=True))
    moved = [m for m in outs if base in os.path.basename(m)]
    for m in moved:
        shutil.copy(m, OUT)
    print(f'   {"OK" if moved else "NO OUTPUT"} ({time.time()-t:.0f}s): {[os.path.basename(m) for m in moved]}')
    if not moved:
        print('   ERR tail:', (r.stderr or r.stdout)[-600:])
print('\nDONE. seg-vert masks on Drive:', len(glob.glob(f'{OUT}/*seg-vert*')))

In [ ]:
# Verify the corpus (49) + endplate (62) labels are present (label IDs can vary by model version)
import nibabel as nib, numpy as np, glob
sp = sorted(glob.glob('/content/cases/**/*seg-spine*.nii.gz', recursive=True))
if sp:
    u = np.unique(np.asarray(nib.load(sp[0]).dataobj))
    print('seg-spine labels:', sorted(int(x) for x in u if x))
    print('corpus(49) present:', 49 in u, '| endplate(62) present:', 62 in u)
else:
    print('no seg-spine output found yet')

In [ ]:
# Zip + download the SPINEPS masks (seg-spine + seg-vert) for local analysis
import shutil
from google.colab import files
shutil.make_archive('/content/spineps_out', 'zip', '/content/drive/MyDrive/spinegeneric_spineps')
files.download('/content/spineps_out.zip')   # -> ~/Downloads/ ; unzip into ~/dev/group5-proto/out_sg_spineps/